# LeetCode 443 — String Compression bằng Python

Link đề: https://leetcode.com/problems/string-compression/ · Độ khó: **Medium** · Chủ đề: Two Pointers, String (sửa mảng tại chỗ)

Skeleton LeetCode:

```python
class Solution:
    def compress(self, chars: List[str]) -> int:
        ...
```

Ghi chú: đề và ví dụ được viết lại theo hiểu biết về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (so với cách nén dùng `itertools.groupby`), nhưng nên đối chiếu lại đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: nén **tại chỗ** một mảng ký tự theo kiểu *run-length*: mỗi nhóm ký tự giống nhau liên tiếp thành "ký tự + số lần" (nếu số lần là `1` thì chỉ giữ ký tự), rồi **trả về độ dài mới**; bài chấm bằng **hai thứ**: độ dài trả về **và** `len` ký tự đầu của `chars`.
2. Nắm các công cụ Python: duyệt mảng bằng chỉ số, hai con trỏ đọc/ghi, `str(số)` rồi `for digit in ...`, lát cắt `chars[:k]`, `itertools.groupby`, phép `divmod`, và đảo một đoạn bằng hai con trỏ.
3. Đi từ **dựng mảng mới bằng `groupby` rồi chép đè lại** (tốn `O(n)` bộ nhớ) → **hai con trỏ đọc/ghi** → **một lượt duyệt, "xả" nhóm khi gặp ký tự khác** → **tự tách chữ số bằng `% 10` rồi đảo ngược** (không dùng `str`).
4. Hiểu mấu chốt: con trỏ **ghi `w` không bao giờ vượt con trỏ đọc `i`**, nên ghi đè lên chỗ đã đọc xong là an toàn; nhóm có đếm `>= 2` luôn được nén thành **không dài hơn** chính nhóm đó.
5. Tránh các lỗi kinh điển: nhét số nhiều chữ số vào **một** ô (`"12"` chứ không phải `"1"`, `"2"`), ghi cả số `1`, quên nhóm cuối, và quên rằng đáp án là **độ dài**, không phải mảng.

## 2. Đề bài & ví dụ

Cho mảng ký tự `chars`. Hãy nén nó theo thuật toán sau: bắt đầu với chuỗi rỗng `s`. Với mỗi **nhóm các ký tự giống nhau liên tiếp** trong `chars`:

- nếu độ dài nhóm là `1`, nối ký tự đó vào `s`;
- ngược lại, nối ký tự đó rồi **độ dài nhóm** vào `s`.

Chuỗi nén `s` **không được trả về riêng**, mà phải được lưu **ngay trong mảng `chars`** (ghi đè từ đầu mảng). Chú ý: độ dài nhóm từ `10` trở lên được **tách thành nhiều ký tự** (`12` thành `'1'`, `'2'`). Sau khi sửa xong mảng, trả về **độ dài mới** của mảng (độ dài chuỗi nén). Chỉ được dùng **bộ nhớ phụ hằng số**. Các ký tự nằm sau độ dài trả về **không quan trọng**.

**Ví dụ 1**

- Input: `chars = ["a","a","b","b","c","c","c"]`
- Output: `6`
- Giải thích: ba nhóm `"aa"`, `"bb"`, `"ccc"` nén thành `"a2b2c3"`; sau khi sửa tại chỗ, `6` ký tự đầu của `chars` là `["a","2","b","2","c","3"]`.

**Ví dụ 2**

- Input: `chars = ["a"]`
- Output: `1`
- Giải thích: nhóm duy nhất là `"a"` (một ký tự) nên giữ nguyên.

**Ví dụ 3**

- Input: `chars = ["a","b","b","b","b","b","b","b","b","b","b","b","b"]`
- Output: `4`
- Giải thích: nhóm `"a"` và nhóm `"bbbbbbbbbbbb"` (12 chữ `b`) nén thành `"ab12"`; `4` ký tự đầu là `["a","b","1","2"]`.

**Ràng buộc:**

- `1 <= chars.length <= 2000`
- `chars[i]` là chữ cái thường, chữ cái hoa, chữ số, hoặc ký hiệu.

Chú ý: vì đề chấm cả nội dung `len` ký tự đầu, trong notebook ta kiểm tra **cả độ dài trả về lẫn `chars[:len]`**.

In [1]:
from typing import List
from itertools import groupby
import random

examples = [
    (["a", "a", "b", "b", "c", "c", "c"], 6, ["a", "2", "b", "2", "c", "3"]),
    (["a"], 1, ["a"]),
    (["a"] + ["b"] * 12, 4, ["a", "b", "1", "2"]),
]
for chars, k, head in examples:
    print("".join(chars), "-> kỳ vọng độ dài", k, "và", head)

aabbccc -> kỳ vọng độ dài 6 và ['a', '2', 'b', '2', 'c', '3']
a -> kỳ vọng độ dài 1 và ['a']
abbbbbbbbbbbb -> kỳ vọng độ dài 4 và ['a', 'b', '1', '2']


## 3. Kiến thức nền

### 3.1. Mảng ký tự, sửa tại chỗ và giá trị trả về

`chars` là một `list` các chuỗi **dài 1 ký tự**. "Sửa tại chỗ" (in-place) nghĩa là ta thay đổi **chính đối tượng list** được truyền vào (gán `chars[w] = ...`), **không** tạo list mới rồi `return` nó. Hàm chỉ trả về một con số: độ dài đoạn đầu mang kết quả.

In [2]:
def demo_inplace(chars):
    chars[0] = "X"          # sửa trực tiếp list của người gọi
    return len(chars)


arr = ["a", "b", "c"]
print(demo_inplace(arr), arr)           # 3 ['X', 'b', 'c']

def demo_rebind(chars):
    chars = ["Z"]           # chỉ gán lại tên cục bộ, list của người gọi không đổi
    return len(chars)


arr = ["a", "b", "c"]
print(demo_rebind(arr), arr)            # 1 ['a', 'b', 'c']

3 ['X', 'b', 'c']
1 ['a', 'b', 'c']


### 3.2. Nhóm các ký tự giống nhau liên tiếp (run) và `itertools.groupby`

Một **run/nhóm** là đoạn tối đại các ký tự bằng nhau đứng liền nhau. Ví dụ `"aabccc"` có ba nhóm: `aa`, `b`, `ccc`. `groupby(chars)` trả về từng cặp `(ký_tự, các_phần_tử_trong_nhóm)`; `len(list(g))` cho độ dài nhóm. Lưu ý: `groupby` chỉ gom các phần tử **liền kề** (khác với `Counter`).

In [3]:
s = "aabccca"
print([(k, len(list(g))) for k, g in groupby(s)])     # [('a', 2), ('b', 1), ('c', 3), ('a', 1)]
# 'a' cuối là một nhóm riêng vì không liền với 'aa' đầu

[('a', 2), ('b', 1), ('c', 3), ('a', 1)]


### 3.3. Đổi số thành các chữ số: `str(n)`

`str(12)` là `"12"`; duyệt chuỗi cho ta từng **ký tự chữ số** `'1'`, `'2'`. Đề bắt ta ghi **mỗi chữ số vào một ô riêng** của `chars`.

In [4]:
count = 12
print(str(count), list(str(count)))      # 12 ['1', '2']
for d in str(count):
    print("ghi vào một ô:", repr(d))
print(len(str(9)), len(str(10)), len(str(2000)))   # 1 2 4

12 ['1', '2']
ghi vào một ô: '1'
ghi vào một ô: '2'
1 2 4


### 3.4. Hai con trỏ đọc/ghi (read/write pointers)

Ta dùng `i` để **đọc** (quét qua mảng) và `w` để **ghi** (vị trí sẽ ghi kết quả). Ý tưởng như "nén mảng tại chỗ": kết quả luôn **ngắn hơn hoặc bằng** phần đã đọc, nên `w <= i` và ta không ghi đè lên dữ liệu chưa đọc.

In [5]:
# Ví dụ nhỏ: xóa số 0 khỏi mảng tại chỗ bằng hai con trỏ
a = [0, 1, 0, 3, 12]
w = 0
for i in range(len(a)):
    if a[i] != 0:
        a[w] = a[i]
        w += 1
print(a[:w], "độ dài", w)       # [1, 3, 12] độ dài 3

[1, 3, 12] độ dài 3


### 3.5. Lát cắt `chars[:k]` và so sánh list

`chars[:k]` lấy `k` phần tử đầu (một bản sao). Hai list so sánh bằng `==` theo từng phần tử. Ta dùng nó để kiểm tra "`len` ký tự đầu" như bài chấm.

In [6]:
arr = ["a", "2", "b", "2", "c", "3", "c"]
print(arr[:6], arr[:6] == ["a", "2", "b", "2", "c", "3"])
print(arr[6:])      # các ký tự sau độ dài trả về không quan trọng

['a', '2', 'b', '2', 'c', '3'] True
['c']


### 3.6. Tách chữ số bằng `divmod` / `% 10` và đảo đoạn

Số `n` có chữ số cuối là `n % 10`, phần còn lại là `n // 10` (cả hai bằng `divmod(n, 10)`). Lặp lại cho đến khi `n == 0` thì ta được **các chữ số từ phải sang trái**, nên phải **đảo ngược** để ra thứ tự đọc. Đây là cách làm không cần `str`.

In [7]:
print(divmod(123, 10), divmod(12, 10), divmod(1, 10))     # (12, 3) (1, 2) (0, 1)

n = 1203
digits = []
while n > 0:
    n, d = divmod(n, 10)
    digits.append(d)
print(digits, "-> đảo:", digits[::-1])                    # [3, 0, 2, 1] -> [1, 2, 0, 3]

(12, 3) (1, 2) (0, 1)
[3, 0, 2, 1] -> đảo: [1, 2, 0, 3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy nén `"aabbbbbbbbbbbbc"` (hai chữ `a`, mười hai chữ `b`, một chữ `c`) bằng tay:

1. Nhóm 1: `a` lặp `2` lần → ghi `a2`.
2. Nhóm 2: `b` lặp `12` lần → ghi `b12` (tức `b`, `1`, `2`).
3. Nhóm 3: `c` lặp `1` lần → chỉ ghi `c`.

Kết quả `"a2b12c"`, dài `6`, so với `15` ban đầu.

Cách nghĩ: **quét từ trái sang phải, tìm ranh giới từng nhóm; mỗi nhóm viết ra `ký tự` (+ `số lần` nếu `>= 2`)**. Điều khó duy nhất là viết **ngay trong chính mảng đang đọc**: ta phải chắc chắn rằng chỗ ta viết đã được đọc xong.

In [8]:
s = "aabbbbbbbbbbbbc"
out = []
for key, group in groupby(s):
    run = len(list(group))
    out.append(key)
    if run > 1:
        out.append(str(run))
    print(f"nhóm {key!r} dài {run} -> thêm {key + (str(run) if run > 1 else '')!r}")
print("".join(out), len("".join(out)))

nhóm 'a' dài 2 -> thêm 'a2'
nhóm 'b' dài 12 -> thêm 'b12'
nhóm 'c' dài 1 -> thêm 'c'
a2b12c 6


## 5. Cách 1 — Brute Force / baseline: dựng mảng mới rồi chép đè lại

**Ý tưởng.** Dùng `groupby` để lấy từng nhóm, dựng một **list mới** `new` các ký tự nén, rồi **chép đè** vào đầu `chars` bằng gán lát cắt `chars[:len(new)] = new` (vì `len(new) <= len(chars)` nên phần đuôi giữ nguyên, không ảnh hưởng). Trả về `len(new)`.

In [9]:
def compress_groupby(chars):
    new = []
    for key, group in groupby(chars):
        run = len(list(group))
        new.append(key)
        if run > 1:
            new.extend(str(run))        # extend trên chuỗi thêm từng chữ số một
    chars[:len(new)] = new              # ghi đè đầu mảng; độ dài mảng không đổi
    return len(new)


for chars, k, head in examples:
    arr = chars[:]
    got = compress_groupby(arr)
    print(got, arr[:got], "(kỳ vọng", k, head, ")")

6 ['a', '2', 'b', '2', 'c', '3'] (kỳ vọng 6 ['a', '2', 'b', '2', 'c', '3'] )
1 ['a'] (kỳ vọng 1 ['a'] )
4 ['a', 'b', '1', '2'] (kỳ vọng 4 ['a', 'b', '1', '2'] )


**Độ phức tạp (Cách 1).** Time `O(n)`; **Space `O(n)`** vì list `new` (và `list(group)` tạm thời). Chạy đúng nhưng **vi phạm** yêu cầu bộ nhớ phụ hằng số của đề — ta dùng nó làm **oracle** để đối chiếu.

**Nút thắt:** mảng phụ `new`. Ta cần ghi kết quả **trực tiếp** vào `chars` khi đang đọc nó.

## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát khiến việc ghi tại chỗ trở nên an toàn:

1. **Nhóm dài `count` luôn nén thành không quá `count` ký tự.** Nhóm `count = 1` → `1` ký tự. Nhóm `count >= 2` → `1 + (số chữ số của count)` ký tự: `count = 2` → `2` ký tự; `count = 10` → `3`; `count = 2000` → `5`. Luôn `<= count`.
2. Vì vậy nếu `w` là vị trí ghi tiếp theo và `i` là đầu nhóm đang xét, ta luôn có `w <= i`; khi ghi kết quả của nhóm `[i, j)`, ta chỉ ghi vào các ô `< j`, đều **đã được đọc**.

Từ đó cách làm: **hai con trỏ**. `i` đứng ở đầu nhóm, đẩy `j` đến hết nhóm để biết `count = j - i`; ghi ký tự và chữ số tại `w`; rồi `i = j`. Ba biến thể ở mục 7: tìm `j` bằng vòng lặp trong, xả nhóm khi gặp ký tự khác (một lượt), và tự tách chữ số không dùng `str`.

## 7. Các cách tiếp theo

Tổng cộng ta có **4 cách**: (1) dựng mảng mới (mục 5, oracle); (2) hai con trỏ + vòng lặp trong; (3) một lượt, xả nhóm khi gặp ký tự khác; (4) tách chữ số thủ công rồi đảo. Mỗi cách đều: giải thích → code demo → code nộp LeetCode → độ phức tạp.

### 7.1. Cách 2 — Hai con trỏ, tìm hết nhóm bằng vòng lặp trong

**Giải thích từng đoạn.**

- `i = 0` (đầu nhóm), `w = 0` (vị trí ghi). `while i < n`: đây là vòng lặp ngoài, mỗi vòng xử lý **một nhóm**.
- `j = i`; `while j < n and chars[j] == chars[i]: j += 1`. Khi dừng, nhóm là `chars[i:j]` và `count = j - i`.
- Ghi ký tự: `chars[w] = chars[i]; w += 1`.
- Nếu `count > 1`: `for d in str(count): chars[w] = d; w += 1`.
- `i = j` sang nhóm kế tiếp. Cuối cùng `return w`.

Thứ tự **đọc `chars[i]` trước khi ghi** quan trọng: `chars[w] = chars[i]` lấy ký tự rồi mới ghi (và `w <= i` nên ghi tại `w` không làm hỏng các ký tự `chars[j...]` chưa đọc, kể cả khi `w == i`).

In [10]:
def compress_two_pointers(chars):
    n = len(chars)
    i = w = 0
    while i < n:
        j = i
        while j < n and chars[j] == chars[i]:
            j += 1
        count = j - i
        chars[w] = chars[i]
        w += 1
        if count > 1:
            for d in str(count):
                chars[w] = d
                w += 1
        i = j
    return w


for chars, k, head in examples:
    arr = chars[:]
    got = compress_two_pointers(arr)
    print(got, arr[:got], "(kỳ vọng", k, head, ")")

6 ['a', '2', 'b', '2', 'c', '3'] (kỳ vọng 6 ['a', '2', 'b', '2', 'c', '3'] )
1 ['a'] (kỳ vọng 1 ['a'] )
4 ['a', 'b', '1', '2'] (kỳ vọng 4 ['a', 'b', '1', '2'] )


In [11]:
class SolutionTwoPointers:
    def compress(self, chars: List[str]) -> int:
        n = len(chars)
        i = w = 0
        while i < n:
            j = i
            while j < n and chars[j] == chars[i]:
                j += 1
            count = j - i
            chars[w] = chars[i]
            w += 1
            if count > 1:
                for d in str(count):
                    chars[w] = d
                    w += 1
            i = j
        return w


arr = ["a", "a", "b", "b", "c", "c", "c"]
k = SolutionTwoPointers().compress(arr)
print(k, arr[:k])

6 ['a', '2', 'b', '2', 'c', '3']


**Độ phức tạp (Cách 2).** Mỗi phần tử được đọc đúng **một lần** (`j` chỉ tiến, `i = j` sau mỗi nhóm) ⇒ **Time `O(n)`**. Bộ nhớ phụ: `str(count)` có tối đa `4` ký tự (vì `count <= 2000`), tức **Space `O(1)`** (nói chặt hơn: `O(log n)` cho chuỗi chữ số, hằng số với ràng buộc của đề).

### 7.2. Cách 3 — Một lượt duyệt, xả (flush) nhóm khi gặp ký tự khác

**Giải thích từng đoạn.**

- Không có vòng lặp trong. Dùng biến `count` = độ dài nhóm hiện tại và con trỏ ghi `w`.
- Với mỗi chỉ số `i`, tăng `count += 1`.
- Nếu `i` là ký tự cuối của mảng **hoặc** `chars[i] != chars[i + 1]`: nhóm kết thúc → **xả**: ghi `chars[i]` rồi các chữ số của `count` (nếu `count > 1`), rồi đặt `count = 0`.

Điểm cần nhớ: ký tự cần ghi là `chars[i]` (đang là ký tự của nhóm vừa kết thúc), và phép kiểm tra "là ký tự cuối" phải đặt **trước** `chars[i + 1]` để không vượt biên. Nhóm **cuối cùng** được xả nhờ điều kiện `i == n - 1`.

Vì sao ghi khi xả là an toàn? Lúc xả tại `i`, ta đã đọc xong mọi ký tự từ đầu mảng đến `i` (tức `i + 1` ký tự). Tổng số ký tự đã ghi (kể cả mã của nhóm này) không vượt quá số ký tự đã đọc, nên mọi ô được ghi đều có chỉ số `<= i` — những ô đã đọc xong.

In [12]:
def compress_flush(chars):
    n = len(chars)
    w = 0
    count = 0
    for i in range(n):
        count += 1
        if i == n - 1 or chars[i] != chars[i + 1]:
            chars[w] = chars[i]
            w += 1
            if count > 1:
                for d in str(count):
                    chars[w] = d
                    w += 1
            count = 0
    return w


for chars, k, head in examples:
    arr = chars[:]
    got = compress_flush(arr)
    print(got, arr[:got], "(kỳ vọng", k, head, ")")

6 ['a', '2', 'b', '2', 'c', '3'] (kỳ vọng 6 ['a', '2', 'b', '2', 'c', '3'] )
1 ['a'] (kỳ vọng 1 ['a'] )
4 ['a', 'b', '1', '2'] (kỳ vọng 4 ['a', 'b', '1', '2'] )


In [13]:
class SolutionFlush:
    def compress(self, chars: List[str]) -> int:
        n = len(chars)
        w = 0
        count = 0
        for i in range(n):
            count += 1
            if i == n - 1 or chars[i] != chars[i + 1]:
                chars[w] = chars[i]
                w += 1
                if count > 1:
                    for d in str(count):
                        chars[w] = d
                        w += 1
                count = 0
        return w


arr = ["a"] + ["b"] * 12
k = SolutionFlush().compress(arr)
print(k, arr[:k])

4 ['a', 'b', '1', '2']


**Độ phức tạp (Cách 3).** Một vòng `for` duy nhất, mỗi phần tử một lần: **Time `O(n)`**, **Space `O(1)`**.

### 7.3. Cách 4 — Tự tách chữ số bằng `% 10`, rồi đảo đoạn

**Giải thích từng đoạn.** Giống Cách 2, nhưng **không dùng `str`**. Ghi chữ số của `count` như sau:

1. Nhớ `start = w` (vị trí bắt đầu ghi chữ số).
2. Trong khi `count > 0`: ghi `chr(ord('0') + count % 10)` (chữ số cuối) vào `chars[w]`, `w += 1`, `count //= 10`. Chữ số được ghi **từ phải sang trái**.
3. **Đảo** đoạn `chars[start:w]` bằng hai con trỏ `lo = start`, `hi = w - 1` để về đúng thứ tự đọc.

Tại sao cần đảo? Vì `12 % 10 = 2` được ghi trước, rồi `1`; đoạn là `['2', '1']`, đảo thành `['1', '2']`.

In [14]:
def reverse_segment(a, lo, hi):
    while lo < hi:
        a[lo], a[hi] = a[hi], a[lo]
        lo += 1
        hi -= 1


def compress_digits(chars):
    n = len(chars)
    i = w = 0
    while i < n:
        j = i
        while j < n and chars[j] == chars[i]:
            j += 1
        count = j - i
        chars[w] = chars[i]
        w += 1
        if count > 1:
            start = w
            while count > 0:
                chars[w] = chr(ord("0") + count % 10)
                w += 1
                count //= 10
            reverse_segment(chars, start, w - 1)
        i = j
    return w


for chars, k, head in examples:
    arr = chars[:]
    got = compress_digits(arr)
    print(got, arr[:got], "(kỳ vọng", k, head, ")")

6 ['a', '2', 'b', '2', 'c', '3'] (kỳ vọng 6 ['a', '2', 'b', '2', 'c', '3'] )
1 ['a'] (kỳ vọng 1 ['a'] )
4 ['a', 'b', '1', '2'] (kỳ vọng 4 ['a', 'b', '1', '2'] )


In [15]:
class SolutionDigits:
    def compress(self, chars: List[str]) -> int:
        n = len(chars)
        i = w = 0
        while i < n:
            j = i
            while j < n and chars[j] == chars[i]:
                j += 1
            count = j - i
            chars[w] = chars[i]
            w += 1
            if count > 1:
                start = w
                while count > 0:
                    chars[w] = chr(ord("0") + count % 10)
                    w += 1
                    count //= 10
                lo, hi = start, w - 1
                while lo < hi:
                    chars[lo], chars[hi] = chars[hi], chars[lo]
                    lo += 1
                    hi -= 1
            i = j
        return w


arr = ["a"] + ["b"] * 12
k = SolutionDigits().compress(arr)
print(k, arr[:k])

4 ['a', 'b', '1', '2']


**Độ phức tạp (Cách 4).** Với mỗi nhóm, tách chữ số tốn `O(log count)` và đảo tốn cùng bậc: tổng **Time `O(n)`** (vì `Σ log count <= Σ count = n`), **Space `O(1)`** thật sự (không có chuỗi phụ).

## 8. TRACE từng bước

### 8.1. Bảng trạng thái — Cách 2 với `["a","a","b","b","c","c","c"]` (`n = 7`)

| Vòng ngoài | `i` | `j` cuối | `count` | Ghi | `w` sau | `chars[:w]` |
|---|---|---|---|---|---|---|
| 1 | 0 | 2 | 2 | `a`, `2` | 2 | `a 2` |
| 2 | 2 | 4 | 2 | `b`, `2` | 4 | `a 2 b 2` |
| 3 | 4 | 7 | 3 | `c`, `3` | 6 | `a 2 b 2 c 3` |

Kết thúc: `i = 7 = n`, trả về `w = 6`. Để ý ở vòng 3 ta ghi vào ô `4` và `5`, trong khi vẫn còn các ký tự `c` ở ô `4, 5, 6`: **đúng lúc ghi, ta đã đọc xong cả nhóm** (`j = 7`), nên không mất dữ liệu.

In [16]:
def trace_two_pointers(chars):
    chars = chars[:]
    n = len(chars)
    i = w = 0
    print("bắt đầu:", chars, "n =", n)
    round_no = 0
    while i < n:
        round_no += 1
        j = i
        while j < n and chars[j] == chars[i]:
            j += 1
        count = j - i
        print(f"vòng {round_no}: i={i} ký tự={chars[i]!r} -> j={j}, count={count}")
        chars[w] = chars[i]
        w += 1
        if count > 1:
            for d in str(count):
                chars[w] = d
                w += 1
        print(f"         w={w}, chars[:w]={chars[:w]}, toàn mảng={chars}")
        i = j
    return w, chars[:w]


print(trace_two_pointers(["a", "a", "b", "b", "c", "c", "c"]))

bắt đầu: ['a', 'a', 'b', 'b', 'c', 'c', 'c'] n = 7
vòng 1: i=0 ký tự='a' -> j=2, count=2
         w=2, chars[:w]=['a', '2'], toàn mảng=['a', '2', 'b', 'b', 'c', 'c', 'c']
vòng 2: i=2 ký tự='b' -> j=4, count=2
         w=4, chars[:w]=['a', '2', 'b', '2'], toàn mảng=['a', '2', 'b', '2', 'c', 'c', 'c']
vòng 3: i=4 ký tự='c' -> j=7, count=3
         w=6, chars[:w]=['a', '2', 'b', '2', 'c', '3'], toàn mảng=['a', '2', 'b', '2', 'c', '3', 'c']
(6, ['a', '2', 'b', '2', 'c', '3'])


### 8.2. Bảng trạng thái — Cách 3 (xả khi gặp ký tự khác) với `["a","a","b","c"]`

| `i` | `chars[i]` | `count` sau `+= 1` | Có xả? (`i == n-1` hoặc `chars[i] != chars[i+1]`) | Ghi | `w` |
|---|---|---|---|---|---|
| 0 | a | 1 | không (`a == a`) | | 0 |
| 1 | a | 2 | **có** (`a != b`) | `a`, `2` | 2 |
| 2 | b | 1 | **có** (`b != c`) | `b` | 3 |
| 3 | c | 1 | **có** (`i` là ký tự cuối) | `c` | 4 |

Kết quả `["a","2","b","c"]`, độ dài `4`.

In [17]:
def trace_flush(chars):
    chars = chars[:]
    n = len(chars)
    w = 0
    count = 0
    for i in range(n):
        count += 1
        end = i == n - 1 or chars[i] != chars[i + 1]
        line = f"i={i} ký tự={chars[i]!r} count={count} xả={end}"
        if end:
            chars[w] = chars[i]
            w += 1
            if count > 1:
                for d in str(count):
                    chars[w] = d
                    w += 1
            count = 0
            line += f" -> w={w}, chars[:w]={chars[:w]}"
        print(line)
    return w, chars[:w]


print(trace_flush(["a", "a", "b", "c"]))
print(trace_flush(["a"] + ["b"] * 12))

i=0 ký tự='a' count=1 xả=False
i=1 ký tự='a' count=2 xả=True -> w=2, chars[:w]=['a', '2']
i=2 ký tự='b' count=1 xả=True -> w=3, chars[:w]=['a', '2', 'b']
i=3 ký tự='c' count=1 xả=True -> w=4, chars[:w]=['a', '2', 'b', 'c']
(4, ['a', '2', 'b', 'c'])
i=0 ký tự='a' count=1 xả=True -> w=1, chars[:w]=['a']
i=1 ký tự='b' count=1 xả=False
i=2 ký tự='b' count=2 xả=False
i=3 ký tự='b' count=3 xả=False
i=4 ký tự='b' count=4 xả=False
i=5 ký tự='b' count=5 xả=False
i=6 ký tự='b' count=6 xả=False
i=7 ký tự='b' count=7 xả=False
i=8 ký tự='b' count=8 xả=False
i=9 ký tự='b' count=9 xả=False
i=10 ký tự='b' count=10 xả=False
i=11 ký tự='b' count=11 xả=False
i=12 ký tự='b' count=12 xả=True -> w=4, chars[:w]=['a', 'b', '1', '2']
(4, ['a', 'b', '1', '2'])


### 8.3. Trace tách chữ số (Cách 4) cho `count = 12`

Ghi lần lượt `12 % 10 = 2`, rồi `count = 1`; ghi `1 % 10 = 1`, `count = 0`. Đoạn ghi được là `['2', '1']` rồi đảo thành `['1', '2']`.

In [18]:
count = 12
seg = []
while count > 0:
    seg.append(chr(ord("0") + count % 10))
    count //= 10
    print("đoạn đang ghi:", seg, "| count còn", count)
seg.reverse()
print("sau khi đảo:", seg)

đoạn đang ghi: ['2'] | count còn 1
đoạn đang ghi: ['2', '1'] | count còn 0
sau khi đảo: ['1', '2']


## 9. Code nộp LeetCode

Ta nộp **Cách 3 (một lượt, xả nhóm khi gặp ký tự khác)**: ngắn, không có vòng lặp trong, `O(n)` thời gian, `O(1)` bộ nhớ phụ.

```python
class Solution:
    def compress(self, chars: List[str]) -> int:
        n = len(chars)
        w = 0
        count = 0
        for i in range(n):
            count += 1
            if i == n - 1 or chars[i] != chars[i + 1]:
                chars[w] = chars[i]
                w += 1
                if count > 1:
                    for d in str(count):
                        chars[w] = d
                        w += 1
                count = 0
        return w
```

In [19]:
class Solution:
    def compress(self, chars: List[str]) -> int:
        n = len(chars)
        w = 0
        count = 0
        for i in range(n):
            count += 1
            if i == n - 1 or chars[i] != chars[i + 1]:
                chars[w] = chars[i]
                w += 1
                if count > 1:
                    for d in str(count):
                        chars[w] = d
                        w += 1
                count = 0
        return w


for chars, k, head in examples:
    arr = chars[:]
    got = Solution().compress(arr)
    print(got, arr[:got])
    assert got == k and arr[:got] == head

6 ['a', '2', 'b', '2', 'c', '3']
1 ['a']
4 ['a', 'b', '1', '2']


## 10. Vì sao đúng

**Đúng nội dung.** Xét nhóm `chars[i:j]` có `count = j - i`. Cách 2 ghi đúng `chars[i]` rồi (nếu `count > 1`) các chữ số thập phân của `count` theo thứ tự từ trái sang phải — đúng đặc tả. Các nhóm được xử lý lần lượt từ trái sang phải và `w` chỉ tăng, nên chuỗi ghi ra là nối các mã nén của từng nhóm theo đúng thứ tự.

**Không ghi đè lên dữ liệu chưa đọc.** Tại thời điểm ghi mã của nhóm `[i, j)`: tổng số ký tự đã ghi (kể cả nhóm này) bằng `Σ len(mã nhóm)`; mà `len(mã) <= count` cho mọi nhóm (đã chứng minh ở mục 6), nên số ký tự đã ghi `<=` số ký tự đã đọc `= j`. Vậy mọi ô ghi đều có chỉ số `< j`, tức đã đọc xong. Cũng vì thế, ghi `chars[w] = chars[i]` lúc `w == i` (nhóm đầu) vô hại.

**Đúng cho nhóm cuối.** Cách 2: vòng `while i < n` xử lý đến hết (đoạn `j` dừng ở `n`). Cách 3: điều kiện `i == n - 1` ép xả nhóm cuối.

**Đúng khi count có nhiều chữ số.** Mỗi chữ số được ghi vào **một ô riêng** (`for d in str(count)`), khớp quy ước của đề; trả về `w` là độ dài chuỗi nén.

**Dừng.** Cách 2: `i = j > i` mỗi vòng (vì nhóm có ít nhất một phần tử). Cách 3: vòng `for` hữu hạn.

## 11. Time / Space Complexity

- **Time `O(n)`**: mỗi phần tử của `chars` được đọc một lần; việc ghi chữ số của `count` tốn `O(log count)`, tổng `<= O(n)`.
- **Space `O(1)`** phụ: chỉ vài biến (`i`, `j`, `w`, `count`, một chuỗi chữ số tối đa `4` ký tự với `n <= 2000`). Cách 1 (dựng mảng mới) là `O(n)`.

Với `n <= 2000` mọi cách `O(n)` chạy tức thì; ràng buộc bộ nhớ hằng số là điểm khác biệt chính giữa các cách.

## 12. Bộ test

Bài này **sửa tại chỗ**, nên ta kiểm tra **cả hai thứ**: độ dài trả về và `chars[:len]`. Oracle độc lập: dựng chuỗi nén bằng `groupby` (không đụng vào mảng gốc). Mỗi lần gọi một cách ta truyền **bản sao** của input.

In [20]:
def expected_of(chars):
    out = []
    for key, group in groupby(chars):
        run = len(list(group))
        out.append(key)
        if run > 1:
            out.extend(str(run))
    return out


impls = {
    "groupby": compress_groupby,
    "two_pointers": compress_two_pointers,
    "flush": compress_flush,
    "digits": compress_digits,
    "leetcode": lambda a: Solution().compress(a),
    "submit_two": lambda a: SolutionTwoPointers().compress(a),
    "submit_flush": lambda a: SolutionFlush().compress(a),
    "submit_digits": lambda a: SolutionDigits().compress(a),
}


def check_all(chars, label, expected_head=None):
    exp = expected_of(chars)
    if expected_head is not None:
        assert exp == expected_head, (label, "oracle", exp, expected_head)
    for key, fn in impls.items():
        arr = chars[:]
        k = fn(arr)
        assert k == len(exp), (label, key, "độ dài", k, len(exp))
        assert arr[:k] == exp, (label, key, "nội dung", arr[:k], exp)
    print("PASS", label, "->", "".join(exp)[:40], "| độ dài", len(exp))


for idx, (chars, k, head) in enumerate(examples, 1):
    check_all(chars, f"ví dụ {idx}", head)
    assert len(head) == k

PASS ví dụ 1 -> a2b2c3 | độ dài 6
PASS ví dụ 2 -> a | độ dài 1
PASS ví dụ 3 -> ab12 | độ dài 4


### 12.1. Các trường hợp biên

In [21]:
check_all(["a"], "một ký tự")
check_all(["a", "a"], "hai ký tự giống", ["a", "2"])
check_all(["a", "b"], "hai ký tự khác", ["a", "b"])
check_all(list("abcdef"), "toàn khác nhau (không nén được)", list("abcdef"))
check_all(["z"] * 9, "nhóm 9 (1 chữ số)", ["z", "9"])
check_all(["z"] * 10, "nhóm 10 (2 chữ số)", ["z", "1", "0"])
check_all(["z"] * 99, "nhóm 99", ["z", "9", "9"])
check_all(["z"] * 100, "nhóm 100 (3 chữ số)", ["z", "1", "0", "0"])
check_all(["z"] * 1000, "nhóm 1000", ["z", "1", "0", "0", "0"])
check_all(["z"] * 2000, "toàn bộ 2000 chữ giống nhau", ["z", "2", "0", "0", "0"])
check_all(list("aabbcc"), "hai nhóm nhỏ liên tiếp", list("a2b2c2"))
check_all(list("abbbc"), "đơn - nhóm - đơn", list("ab3c"))
check_all(list("aabcc"), "nhóm - đơn - nhóm", list("a2bc2"))
check_all(list("ababab"), "xen kẽ, không nhóm", list("ababab"))
check_all(["1"] * 12, "ký tự là chữ số", ["1", "1", "2"])
check_all(["#", "#", "#", "$", "$", " ", " ", " ", " "], "ký hiệu và khoảng trắng", ["#", "3", "$", "2", " ", "4"])
check_all(list("aaabbbcccd"), "nhiều nhóm", list("a3b3c3d"))
check_all(list("Aa" * 5), "chữ hoa/thường khác nhau", list("Aa" * 5))

PASS một ký tự -> a | độ dài 1
PASS hai ký tự giống -> a2 | độ dài 2
PASS hai ký tự khác -> ab | độ dài 2
PASS toàn khác nhau (không nén được) -> abcdef | độ dài 6
PASS nhóm 9 (1 chữ số) -> z9 | độ dài 2
PASS nhóm 10 (2 chữ số) -> z10 | độ dài 3
PASS nhóm 99 -> z99 | độ dài 3
PASS nhóm 100 (3 chữ số) -> z100 | độ dài 4
PASS nhóm 1000 -> z1000 | độ dài 5
PASS toàn bộ 2000 chữ giống nhau -> z2000 | độ dài 5
PASS hai nhóm nhỏ liên tiếp -> a2b2c2 | độ dài 6
PASS đơn - nhóm - đơn -> ab3c | độ dài 4
PASS nhóm - đơn - nhóm -> a2bc2 | độ dài 5
PASS xen kẽ, không nhóm -> ababab | độ dài 6
PASS ký tự là chữ số -> 112 | độ dài 3
PASS ký hiệu và khoảng trắng -> #3$2 4 | độ dài 6
PASS nhiều nhóm -> a3b3c3d | độ dài 7
PASS chữ hoa/thường khác nhau -> AaAaAaAaAa | độ dài 10


### 12.2. Phản ví dụ cho các cách làm sai

Ba lỗi: (a) nhét `str(count)` vào **một** ô (số nhiều chữ số thành chuỗi dài hơn 1 ký tự); (b) **ghi cả số `1`** cho nhóm một ký tự; (c) **quên xả nhóm cuối** ở kiểu duyệt một lượt.

In [22]:
def wrong_one_cell(chars):
    n = len(chars)
    i = w = 0
    while i < n:
        j = i
        while j < n and chars[j] == chars[i]:
            j += 1
        count = j - i
        chars[w] = chars[i]
        w += 1
        if count > 1:
            chars[w] = str(count)        # SAI: "12" nằm trong một ô
            w += 1
        i = j
    return w


def wrong_write_one(chars):
    n = len(chars)
    i = w = 0
    while i < n:
        j = i
        while j < n and chars[j] == chars[i]:
            j += 1
        count = j - i
        chars[w] = chars[i]
        w += 1
        for d in str(count):             # SAI: không loại trường hợp count == 1
            chars[w] = d
            w += 1
        i = j
    return w


def wrong_no_last(chars):
    n = len(chars)
    w = 0
    count = 1
    for i in range(1, n):
        if chars[i] == chars[i - 1]:
            count += 1
        else:                            # chỉ xả khi gặp ký tự khác -> nhóm cuối không bao giờ được xả
            chars[w] = chars[i - 1]
            w += 1
            if count > 1:
                for d in str(count):
                    chars[w] = d
                    w += 1
            count = 1
    return w


for name, fn, sample in [
    ("một ô", wrong_one_cell, ["a"] + ["b"] * 12),
    ("ghi số 1", wrong_write_one, list("aaabcc")),
    ("quên nhóm cuối", wrong_no_last, list("aabbb")),
]:
    arr = sample[:]
    k = fn(arr)
    exp = expected_of(sample)
    print(f"{name:>16}: k={k} {arr[:k]} | đúng: {len(exp)} {exp}")
    assert (k, arr[:k]) != (len(exp), exp)

           một ô: k=3 ['a', 'b', '12'] | đúng: 4 ['a', 'b', '1', '2']
        ghi số 1: k=6 ['a', '3', 'b', '1', 'c', '2'] | đúng: 5 ['a', '3', 'b', 'c', '2']
  quên nhóm cuối: k=2 ['a', '2'] | đúng: 4 ['a', '2', 'b', '3']


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh `chars` đúng ràng buộc: nối nhiều nhóm với độ dài ngẫu nhiên (cả nhóm `1`, nhóm vài chục, vài trăm), ký tự từ các bảng nhỏ (để nhóm kề nhau có thể trùng ký tự → gộp lại thành một nhóm dài hơn — oracle `groupby` xử lý đúng). Tổng độ dài `<= 2000`. So **độ dài và nội dung** của mọi cách với oracle.

In [23]:
def gen(rng):
    alpha = rng.choice(["a", "ab", "abc", "aA1#", "abcdefghijklmnopqrstuvwxyz0123456789"])
    mode = rng.choice(["small", "medium", "large"])
    limit = {"small": 4, "medium": 30, "large": 400}[mode]
    parts = []
    total = 0
    for _ in range(rng.randint(1, 12)):
        run = rng.randint(1, limit)
        if total + run > 2000:
            break
        parts.append(rng.choice(alpha) * run)
        total += run
    if not parts:
        parts = [rng.choice(alpha)]
    return list("".join(parts))


rng = random.Random(443)
longest_run = 0
for trial in range(4000):
    chars = gen(rng)
    assert 1 <= len(chars) <= 2000
    exp = expected_of(chars)
    longest_run = max(longest_run, max(len(list(g)) for _, g in groupby(chars)))
    for key, fn in impls.items():
        arr = chars[:]
        k = fn(arr)
        assert k == len(exp) and arr[:k] == exp, (key, "".join(chars))
print("4000 test ngẫu nhiên OK; nhóm dài nhất từng gặp:", longest_run)

4000 test ngẫu nhiên OK; nhóm dài nhất từng gặp: 1999


### 12.4. Tính chất: kết quả không dài hơn input, và giải nén khôi phục input

Thuộc tính thứ hai là một kiểm tra độc lập với oracle `groupby`: nếu ta **giải nén** (đọc `ký tự`, rồi các chữ số đứng liền sau) thì phải được đúng chuỗi ban đầu — **miễn là ký tự gốc không phải chữ số** (nếu không, giải nén bị nhập nhằng). Ta kiểm tra trên bảng chữ cái không chứa chữ số.

In [24]:
def decompress(head):
    out = []
    i = 0
    while i < len(head):
        ch = head[i]
        i += 1
        num = ""
        while i < len(head) and head[i].isdigit():
            num += head[i]
            i += 1
        out.append(ch * (int(num) if num else 1))
    return "".join(out)


rng = random.Random(4430)
for trial in range(1500):
    alpha = rng.choice(["a", "ab", "abcXYZ#"])
    chars = list("".join(rng.choice(alpha) * rng.randint(1, 40) for _ in range(rng.randint(1, 10))))
    arr = chars[:]
    k = Solution().compress(arr)
    assert k <= len(chars)
    assert decompress(arr[:k]) == "".join(chars), "".join(chars)
print("giải nén khôi phục đúng input, độ dài nén luôn <= độ dài gốc")

giải nén khôi phục đúng input, độ dài nén luôn <= độ dài gốc


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Nhét cả số `count` vào một ô

**Code sai:**

```python
chars[w] = str(count)
```

**Vì sao sai:** với `count = 12`, ô đó chứa chuỗi `"12"` (dài 2) trong khi đề muốn **hai ô** `'1'`, `'2'`. Độ dài trả về cũng thiếu một và `chars[:len]` không khớp.

**Code đúng:** `for d in str(count): chars[w] = d; w += 1`.

In [25]:
arr = ["a"] + ["b"] * 12
k = wrong_one_cell(arr)
print("sai:", k, arr[:k])
arr = ["a"] + ["b"] * 12
k = Solution().compress(arr)
print("đúng:", k, arr[:k])

sai: 3 ['a', 'b', '12']
đúng: 4 ['a', 'b', '1', '2']


### Lỗi 2: Ghi số `1` cho nhóm một ký tự

**Code sai:** bỏ điều kiện `if count > 1` (ghi luôn các chữ số của `count`).

**Vì sao sai:** nhóm `"a"` thành `"a1"` — dài hơn và khác đặc tả ("độ dài nhóm là `1` thì chỉ giữ ký tự"). Tệ hơn: nếu mọi nhóm đều đơn, chuỗi nén dài **gấp đôi** gốc và `w` vượt `n`, gây `IndexError` do ghi ra ngoài mảng.

**Code đúng:** chỉ ghi chữ số khi `count > 1`.

In [26]:
arr = list("abc")
try:
    k = wrong_write_one(arr)
    print("sai:", k, arr[:k])
except IndexError as e:
    print("IndexError:", e)

arr = list("abcd")
try:
    k = wrong_write_one(arr)
    print("sai:", k, arr[:k])
except IndexError as e:
    print("IndexError (ghi vượt biên):", e)

IndexError: list assignment index out of range
IndexError (ghi vượt biên): list assignment index out of range


### Lỗi 3: Quên xả nhóm cuối

**Code sai:** duyệt `i` từ `1`, so `chars[i]` với `chars[i - 1]`, chỉ ghi khi gặp ký tự **khác** — vòng lặp kết thúc mà nhóm cuối chưa được ghi.

**Vì sao sai:** ví dụ `"aabbb"`: gặp `b` khác `a` ở `i = 2` thì xả `a2`; còn `bbb` không bao giờ gặp ký tự khác nên **bị bỏ**. Mẹo: sau vòng lặp phải xả thêm một lần, hoặc dùng điều kiện `i == n - 1` như lời giải chính.

In [27]:
arr = list("aabbb")
k = wrong_no_last(arr)
print("sai :", k, arr[:k])
arr = list("aabbb")
k = Solution().compress(arr)
print("đúng:", k, arr[:k])

sai : 2 ['a', '2']
đúng: 4 ['a', '2', 'b', '3']


### Lỗi 4: Trả về mảng thay vì độ dài / tạo mảng mới

**Code sai:**

```python
def compress(self, chars):
    new = [...]
    chars = new           # chỉ gán lại tên cục bộ
    return len(new)
```

**Vì sao sai:** gán `chars = new` **không** thay đổi list mà người gọi đang giữ; bài chấm đọc `chars[:len]` của list gốc nên thấy nội dung cũ. Phải ghi vào chính `chars` (gán từng ô, hoặc gán lát cắt `chars[:k] = new`).

In [28]:
def wrong_rebind(chars):
    new = expected_of(chars)
    chars = new                  # SAI: chỉ đổi tên cục bộ
    return len(new)


arr = ["a", "a", "b", "b", "c", "c", "c"]
k = wrong_rebind(arr)
print("độ dài trả về:", k, "nhưng chars[:k] =", arr[:k], "(vẫn là nội dung cũ)")
assert arr[:k] != ["a", "2", "b", "2", "c", "3"]

độ dài trả về: 6 nhưng chars[:k] = ['a', 'a', 'b', 'b', 'c', 'c'] (vẫn là nội dung cũ)


## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận dạng:** "sửa mảng **tại chỗ**, trả về độ dài mới" + dữ liệu có cấu trúc theo **nhóm liên tiếp** ⇒ hai con trỏ đọc/ghi (`w <= i`).

Bảng nhắc:

| Câu hỏi | Gợi ý |
|---|---|
| Ghi có làm hỏng dữ liệu chưa đọc không? | Chứng minh `w <= i` (số ghi `<=` số đã đọc) |
| Nhóm cuối có bị quên không? | Dùng điều kiện `i == n - 1` hoặc vòng ngoài `while i < n` |
| Có nhiều chữ số? | Ghi mỗi chữ số một ô: `for d in str(count)` |
| Cấm `str`? | `% 10`, `// 10` rồi đảo đoạn |
| Bài có đáp án là "độ dài" | Chấm bằng `arr[:k]`, nên test cả hai |

Bài cùng họ: **26 Remove Duplicates from Sorted Array**, **27 Remove Element**, **283 Move Zeroes**, **38 Count and Say**, **394 Decode String** (bài giải nén), **1531 String Compression II** (khó hơn).

## 15. Biến thể

### 15.1. Giải nén ra chuỗi (bài ngược lại)

Bài ngược lại của 443: cho `["a","2","b","1","2"]` → `"aabbbbbbbbbbbb"`. Đọc `ký tự` rồi các chữ số liền sau (như hàm `decompress` ở mục 12.4). Chú ý bài gốc không nhập nhằng nếu ký tự không phải chữ số.

In [29]:
assert decompress(list("a2b12c")) == "aa" + "b" * 12 + "c"
assert decompress(list("abc")) == "abc"
print(decompress(list("a2b12c")))

aabbbbbbbbbbbbc


### 15.2. Chỉ nén nhóm có độ dài `>= k`

Với ngưỡng `k` (ví dụ `3`), nhóm ngắn hơn `k` được **giữ nguyên các ký tự** (không viết số). Đây là biến thể của Cách 3: khi xả, nếu `count >= k` thì viết `ký tự + số`, ngược lại viết lặp lại `count` lần ký tự đó (vẫn an toàn: nhóm ngắn ghi đúng `count` ký tự bằng số đã đọc, nhóm dài ghi không quá `count` ký tự, nên `w <= i` luôn đúng).

In [30]:
def compress_min_run(chars, k):
    n = len(chars)
    w = 0
    count = 0
    for i in range(n):
        count += 1
        if i == n - 1 or chars[i] != chars[i + 1]:
            ch = chars[i]
            if count >= k:
                chars[w] = ch
                w += 1
                for d in str(count):
                    chars[w] = d
                    w += 1
            else:
                for _ in range(count):
                    chars[w] = ch
                    w += 1
            count = 0
    return w


arr = list("aabbbcccc")
w = compress_min_run(arr, 3)
print("".join(arr[:w]))          # aab3c4
assert "".join(arr[:w]) == "aab3c4"

aab3c4


### 15.3. Giới hạn mỗi số tối đa 9 (tách nhóm)

Nếu ràng buộc là "mỗi số chỉ một chữ số" (tối đa `9`), nhóm `12` được tách thành hai đoạn `9` và `3`, tức `b9b3`. Vòng lặp: `while run > 0: take = min(9, run); viết ký tự (kèm take nếu take > 1); run -= take`. Chú ý trường hợp đoạn dư là `1` (ví dụ nhóm `10` thành `b9` rồi `b`): đoạn dư đơn lẻ thì không viết số.

In [31]:
def compress_cap9(s):
    out = []
    for key, group in groupby(s):
        run = len(list(group))
        while run > 0:
            take = min(9, run)
            out.append(key + (str(take) if take > 1 else ""))
            run -= take
    return "".join(out)


assert compress_cap9("a" + "b" * 12) == "ab9b3"
assert compress_cap9("aab") == "a2b"
print(compress_cap9("a" + "b" * 12))

ab9b3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dựng mảng mới (`groupby`) | Nén ra list phụ rồi chép đè | `O(n)` | `O(n)` | Làm oracle, hiểu đề; không đạt yêu cầu bộ nhớ |
| 2. Hai con trỏ + vòng trong | Tìm ranh giới nhóm bằng `j`, ghi bằng `w` | `O(n)` | `O(1)` | Dễ đọc nhất, dễ chứng minh |
| 3. Một lượt, xả khi gặp khác | Đếm `count`, xả ở ký tự cuối nhóm | `O(n)` | `O(1)` | **Lời giải nộp**: một vòng `for`, ngắn |
| 4. Tách chữ số thủ công | `% 10` rồi đảo đoạn, không `str` | `O(n)` | `O(1)` | Khi cấm `str`/phỏng vấn hỏi sâu |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán độ dài trả về và `chars[:k]` khi gọi `Solution().compress` cho: `list("aaaa")`, `list("abab")`, `list("a" * 10 + "b")`, `list("zzzyyx")`, `list("1112")`.

**Bài 2 (tính tay).** Với `chars = ["x","x","y","z","z","z","z","z","z","z","z","z","z"]` (hai `x`, một `y`, mười `z`): lập bảng từng nhóm (`i`, `j`, `count`, các ký tự ghi, `w`) và nêu kết quả.

**Bài 3 (điền TODO).** Hoàn thành `compress_todo(chars)` theo Cách 3.

In [32]:
def compress_todo(chars):
    n = len(chars)
    w = 0
    count = 0
    for i in range(n):
        count += 1
        # TODO 1: điều kiện xả nhóm: i == n - 1 hoặc chars[i] != chars[i + 1]
        # TODO 2: trong khối xả: chars[w] = chars[i]; w += 1
        # TODO 3: nếu count > 1 thì ghi từng chữ số của count; rồi count = 0
        pass
    return w

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `compress_reverse_digits(chars)` theo Cách 4 (không dùng `str`), **tự viết** hàm đảo đoạn bên trong.

**Bài 5 (biến thể / mini challenge).** (a) Viết `compress_count_first(chars)` nén theo dạng **số trước, ký tự sau** (`"aab"` → `"2ab"`): vì chữ số được ghi trước ký tự nên hãy chứng minh/chỉ ra rằng ghi tại chỗ vẫn an toàn và cài đặt. (b) Viết `compressed_length(chars)` chỉ **tính độ dài** nén mà **không ghi** vào mảng (dùng `O(1)` bộ nhớ), rồi dùng nó để kiểm tra rằng `compressed_length(chars) == Solution().compress(chars[:])`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"aaaa"` → `a4` (độ dài `2`); `"abab"` → `abab` (`4`); `"a"*10+"b"` → `a10b` (`4`); `"zzzyyx"` → `z3y2x` (`5`); `"1112"` → `132` (ký tự `'1'` lặp 3 lần thành `"13"`, rồi `'2'`; độ dài `3`). Bài 3: điều kiện `if i == n - 1 or chars[i] != chars[i + 1]:`.
- **Mức 2 (vừa):** Bài 2: nhóm `xx` → `x2` (`w=2`); `y` → `y` (`w=3`); `zzzzzzzzzz` (10) → `z10` (`w=6`) ⇒ `["x","2","y","z","1","0"]`, độ dài `6`. Bài 4: ghi chữ số bằng `% 10` rồi đảo `chars[start:w]` bằng hai con trỏ.
- **Mức 3 (sâu):** Bài 5(a): ghi **số trước** nghĩa là cần biết `count` trước khi ghi ký tự; Cách 2 đã biết `count = j - i` trước khi ghi nên ghi tại `w <= i` vẫn an toàn (ghi `len(str(count)) + 1 <= count` hoặc `1` ký tự cho nhóm đơn). 5(b): cộng `1 + len(str(count))` cho nhóm `count > 1`, `1` cho nhóm đơn.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
def run(s):
    arr = list(s)
    k = Solution().compress(arr)
    return k, "".join(arr[:k])


# Bài 1
assert run("aaaa") == (2, "a4")
assert run("abab") == (4, "abab")
assert run("a" * 10 + "b") == (4, "a10b")
assert run("zzzyyx") == (5, "z3y2x")
assert run("1112") == (3, "132")

# Bài 2
assert run("xxy" + "z" * 10) == (6, "x2yz10")


# Bài 3
def compress_todo_ans(chars):
    n = len(chars)
    w = 0
    count = 0
    for i in range(n):
        count += 1
        if i == n - 1 or chars[i] != chars[i + 1]:
            chars[w] = chars[i]
            w += 1
            if count > 1:
                for d in str(count):
                    chars[w] = d
                    w += 1
            count = 0
    return w


for chars, k, head in examples:
    arr = chars[:]
    got = compress_todo_ans(arr)
    assert got == k and arr[:got] == head


# Bài 4
def compress_reverse_digits(chars):
    def rev(a, lo, hi):
        while lo < hi:
            a[lo], a[hi] = a[hi], a[lo]
            lo += 1
            hi -= 1

    n = len(chars)
    i = w = 0
    while i < n:
        j = i
        while j < n and chars[j] == chars[i]:
            j += 1
        count = j - i
        chars[w] = chars[i]
        w += 1
        if count > 1:
            start = w
            while count:
                chars[w] = chr(48 + count % 10)
                w += 1
                count //= 10
            rev(chars, start, w - 1)
        i = j
    return w


for chars, k, head in examples:
    arr = chars[:]
    got = compress_reverse_digits(arr)
    assert got == k and arr[:got] == head


# Bài 5
def compress_count_first(chars):
    n = len(chars)
    i = w = 0
    while i < n:
        j = i
        while j < n and chars[j] == chars[i]:
            j += 1
        count = j - i
        ch = chars[i]
        if count > 1:
            for d in str(count):
                chars[w] = d
                w += 1
        chars[w] = ch
        w += 1
        i = j
    return w


arr = list("aab")
k = compress_count_first(arr)
assert arr[:k] == list("2ab")
arr = list("a" + "b" * 12)
k = compress_count_first(arr)
assert arr[:k] == list("a12b")


def compressed_length(chars):
    n = len(chars)
    i = total = 0
    while i < n:
        j = i
        while j < n and chars[j] == chars[i]:
            j += 1
        count = j - i
        total += 1 + (len(str(count)) if count > 1 else 0)
        i = j
    return total


rng = random.Random(4431)
for _ in range(300):
    chars = gen(rng)
    assert compressed_length(chars) == Solution().compress(chars[:])
print("Tất cả đáp án bài tập đều đúng.")

Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (nén tại chỗ, một lượt, xả khi gặp ký tự khác)

```python
class Solution:
    def compress(self, chars: List[str]) -> int:
        n = len(chars)
        w = 0                                    # vị trí ghi
        count = 0                                # độ dài nhóm hiện tại
        for i in range(n):
            count += 1
            if i == n - 1 or chars[i] != chars[i + 1]:   # nhóm kết thúc tại i
                chars[w] = chars[i]
                w += 1
                if count > 1:
                    for d in str(count):         # mỗi chữ số một ô
                        chars[w] = d
                        w += 1
                count = 0
        return w                                 # độ dài mới
```

**Checklist**

- [ ] Ghi vào **chính** `chars` (không gán lại tên), trả về **độ dài** (`w`)?
- [ ] Nhóm một ký tự **không** có số `1`?
- [ ] Số nhiều chữ số được ghi **mỗi chữ số một ô** (`str(count)` rồi duyệt từng ký tự)?
- [ ] Nhóm cuối được xả (`i == n - 1` hoặc `while i < n` đến hết)?
- [ ] Biết lý do `w <= i` nên không ghi đè dữ liệu chưa đọc?
- [ ] Đã test `["a"]`, `["a"]*10`, `["a"]*2000`, toàn khác nhau, ký tự là chữ số, cả **độ dài và `chars[:k]`**?
- [ ] Skeleton đúng `class Solution` / `def compress(self, chars: List[str]) -> int`?